# DeepLog HDFS Anomaly Detection — MLflow Experiments

Полный пайплайн эксперимента с **DeepLog** (LSTM) для обнаружения аномалий в логах HDFS.  
DeepLog — **лучшая модель (PRD)** по результатам Phase 2.

**Результаты DeepLog (из документации):**
- F1 ~0.94, Precision ~0.93, Recall ~0.95 (20K сессий)
- Архитектура: двухслойный LSTM, окно истории 20 событий, nucleus sampling (top-p)
- Данные: HDFS LogHub v1 — 575K сессий, 3.1% аномалий

**Содержание:**
1. Setup & конфигурация MLflow
2. Загрузка данных и EDA
3. Обучение DeepLog с логированием в MLflow
4. Оценка модели — метрики, артефакты
5. Анализ ошибок (FP/FN)
6. Воспроизводимость (multi-seed)
7. Устойчивость — чувствительность к порогу top-p
8. Сводка экспериментов MLflow

In [1]:
# ── Install dependencies (run once) ────────────────────────────────────────
import subprocess, sys

REQUIRED = ['mlflow', 'seaborn']

for pkg in REQUIRED:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'{pkg} installed.')

print('All dependencies ready.')

All dependencies ready.


## ⚙️ Конфигурация пути к данным

Укажи путь к `Event_traces.csv` в ячейке ниже.  
Если файл ещё не загружен — см. инструкции по загрузке.

In [2]:
# ══════════════════════════════════════════════════════════════════════════
# НАСТРОЙКА ПУТИ К ДАННЫМ
# ══════════════════════════════════════════════════════════════════════════
import os, subprocess
from pathlib import Path

REPO_URL  = 'https://github.com/r-artyukhov/22-team-project.git'
REPO_NAME = '22-team-project'

# ── Автоопределение окружения ──────────────────────────────────────────────
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    REPO_DIR = Path('/content') / REPO_NAME

    if not REPO_DIR.exists():
        print(f'Клонирование репозитория в {REPO_DIR} ...')
        result = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print('✓ Репозиторий склонирован')
        else:
            print('✗ Ошибка клонирования:')
            print(result.stderr)
    else:
        print(f'✓ Репозиторий уже есть: {REPO_DIR}')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       capture_output=True)
else:
    # Локальный запуск: ноутбук находится в MLflow/ внутри репозитория
    REPO_DIR = Path(__file__).resolve().parent.parent if '__file__' in dir() \
               else Path('.').resolve().parent

# ── Пути к данным ──────────────────────────────────────────────────────────
DATA_ROOT   = REPO_DIR / 'HDFS_v1' / 'preprocessed'
DATA_PATH   = DATA_ROOT / 'Event_traces.csv'
LABELS_PATH = DATA_ROOT / 'anomaly_label.csv'
TMPL_PATH   = DATA_ROOT / 'HDFS.log_templates.csv'

# ── Проверка ───────────────────────────────────────────────────────────────
print(f'\nREPO_DIR:  {REPO_DIR}')
print(f'DATA_ROOT: {DATA_ROOT}')
all_ok = True
for p in [DATA_PATH, LABELS_PATH, TMPL_PATH]:
    exists = p.exists()
    all_ok = all_ok and exists
    print(f'  {"✓" if exists else "✗ НЕ НАЙДЕН"}  {p.name}')

if not all_ok:
    print()
    print('Если репозиторий приватный, клонируй вручную:')
    print(f'  !git clone {REPO_URL}')
    print('Или смонтируй Google Drive с репозиторием:')
    print("  from google.colab import drive; drive.mount('/content/drive')")
    print("  REPO_DIR = Path('/content/drive/MyDrive/22-team-project')")
else:
    print('\n✓ Все файлы найдены — можно продолжать!')

✓ Репозиторий уже есть: /content/22-team-project

REPO_DIR:  /content/22-team-project
DATA_ROOT: /content/22-team-project/HDFS_v1/preprocessed
  ✗ НЕ НАЙДЕН  Event_traces.csv
  ✗ НЕ НАЙДЕН  anomaly_label.csv
  ✗ НЕ НАЙДЕН  HDFS.log_templates.csv

Если репозиторий приватный, клонируй вручную:
  !git clone https://github.com/r-artyukhov/22-team-project.git
Или смонтируй Google Drive с репозиторием:
  from google.colab import drive; drive.mount('/content/drive')
  REPO_DIR = Path('/content/drive/MyDrive/22-team-project')


In [3]:
# ── Imports ────────────────────────────────────────────────────────────────
import json
import ast
import random
import warnings
import time
from collections import Counter
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient

warnings.filterwarnings('ignore')

print(f'PyTorch:  {torch.__version__}')
print(f'MLflow:   {mlflow.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device:   {DEVICE}')

PyTorch:  2.11.0+cu128
MLflow:   3.13.0
CUDA:     True
Device:   cuda


In [4]:
# ── Paths & writable directories ───────────────────────────────────────────
# DATA_PATH already set in the configuration cell above

# Find a writable directory for MLflow runs and artifacts
for _candidate in ['/content', '/tmp', str(Path('.').resolve())]:
    try:
        _p = Path(_candidate)
        _test = _p / '.write_test'
        _test.touch()
        _test.unlink()
        WRITABLE_BASE = _p
        break
    except Exception:
        continue

MLRUNS_DIR    = WRITABLE_BASE / 'mlruns'
ARTIFACTS_DIR = WRITABLE_BASE / 'deeplog_artifacts'
MLFLOW_URI    = f'file://{MLRUNS_DIR.resolve()}'

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Writable base:  {WRITABLE_BASE.resolve()}')
print(f'MLflow URI:     {MLFLOW_URI}')
print(f'Artifacts dir:  {ARTIFACTS_DIR}')
print(f'Data path:      {DATA_PATH}  (exists={DATA_PATH.exists()})')

Writable base:  /content
MLflow URI:     file:///content/mlruns
Artifacts dir:  /content/deeplog_artifacts
Data path:      /content/22-team-project/HDFS_v1/preprocessed/Event_traces.csv  (exists=False)


In [5]:
# ── Configuration ──────────────────────────────────────────────────────────
@dataclass
class Config:
    """All hyperparameters. Fully logged to MLflow on every run."""
    # Data
    csv_path:     str            = ''      # filled below after DATA_PATH is known
    seed:         int            = 42
    max_sessions: Optional[int]  = None    # None = all; set e.g. 20000 for quick runs

    # Path (event sequence) model
    history_size:    int   = 20
    top_p:           float = 0.997         # nucleus sampling threshold
    event_embed_dim: int   = 16
    path_hidden_dim: int   = 64
    path_num_layers: int   = 2
    path_dropout:    float = 0.1
    path_lr:         float = 1e-3
    path_batch_size: int   = 1024
    path_epochs:     int   = 1

    # Data split (normal sessions only)
    train_ratio: float = 0.8
    valid_ratio: float = 0.1

    device: str = ''   # filled below after DEVICE is known


# Populate runtime defaults (DATA_PATH and DEVICE come from previous cells)
Config.__dataclass_fields__['csv_path'].default = str(DATA_PATH)
Config.__dataclass_fields__['device'].default   = DEVICE

# Quick sanity-check
cfg_default = Config()
print(f'csv_path: {cfg_default.csv_path}')
print(f'device:   {cfg_default.device}')
print(f'top_p:    {cfg_default.top_p}  history: {cfg_default.history_size}  epochs: {cfg_default.path_epochs}')

csv_path: 
device:   
top_p:    0.997  history: 20  epochs: 1


In [6]:
# ── MLflow Setup ───────────────────────────────────────────────────────────
EXPERIMENT_NAME = 'DeepLog_HDFS_Anomaly_Detection'

mlflow.set_tracking_uri(MLFLOW_URI)

try:
    experiment = mlflow.set_experiment(EXPERIMENT_NAME)
    print(f'Tracking URI:    {MLFLOW_URI}')
    print(f'Experiment:      {EXPERIMENT_NAME}')
    print(f'Experiment ID:   {experiment.experiment_id}')
    print(f'\nLaunch UI with:\n  mlflow ui --backend-store-uri {MLRUNS_DIR.resolve()}')
except Exception as e:
    # Fallback: use SQLite backend in the writable directory
    _sqlite_path = WRITABLE_BASE / 'mlflow.db'
    MLFLOW_URI   = f'sqlite:///{_sqlite_path}'
    mlflow.set_tracking_uri(MLFLOW_URI)
    experiment   = mlflow.set_experiment(EXPERIMENT_NAME)
    print(f'[Fallback] SQLite backend: {_sqlite_path}')
    print(f'Experiment ID: {experiment.experiment_id}')

[Fallback] SQLite backend: /content/mlflow.db
Experiment ID: 1


## Utility Functions & Data Classes

In [7]:
# ── Reproducibility seed ───────────────────────────────────────────────────
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ── CSV cell parser ────────────────────────────────────────────────────────
def parse_list_cell(x):
    """Parse '[E5,E22,E5]' or '[0.0, 1.0, 2.0]' from Event_traces.csv."""
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if not s:
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        pass
    if s.startswith('[') and s.endswith(']'):
        inner = s[1:-1].strip()
        if not inner:
            return []
        parsed = []
        for token in inner.split(','):
            token = token.strip()
            if not token:
                continue
            try:
                parsed.append(float(token))
            except ValueError:
                parsed.append(token)
        return parsed
    raise ValueError(f'Cannot parse cell: {x!r}')


# ── Label → binary ─────────────────────────────────────────────────────────
# Event_traces.csv uses: "Success" (normal=0) / "Fail" (anomaly=1)
# anomaly_label.csv uses: "Normal" (0) / "Anomaly" (1)
_NORMAL_LABELS = {'success', 'normal'}

def safe_label_to_binary(label: str) -> int:
    """Return 0 for normal sessions, 1 for anomalies."""
    return 0 if str(label).strip().lower() in _NORMAL_LABELS else 1


# Quick test
assert safe_label_to_binary('Success') == 0
assert safe_label_to_binary('Fail')    == 1
assert safe_label_to_binary('Normal')  == 0
assert safe_label_to_binary('Anomaly') == 1
print('Utility functions OK.')

Utility functions OK.


In [8]:
# ── Data Classes ───────────────────────────────────────────────────────────
class SessionRecord:
    def __init__(self, block_id, label, events, times, latency):
        self.block_id = block_id
        self.label    = label
        self.events   = events
        self.times    = times
        self.latency  = latency


class Vocab:
    PAD = '<PAD>'
    UNK = '<UNK>'

    def __init__(self, tokens: Sequence[str]):
        uniq = [self.PAD, self.UNK] + sorted(set(tokens))
        self.stoi = {t: i for i, t in enumerate(uniq)}
        self.itos = {i: t for t, i in self.stoi.items()}

    def encode(self, token: str) -> int:
        return self.stoi.get(token, self.stoi[self.UNK])

    def decode(self, idx: int) -> str:
        return self.itos.get(idx, self.UNK)

    def __len__(self):
        return len(self.stoi)


print('Data classes defined.')

Data classes defined.


In [9]:
# ── Data Loading ───────────────────────────────────────────────────────────
def load_sessions(csv_path: str, max_sessions: Optional[int] = None,
                  seed: int = 42) -> List[SessionRecord]:
    df = pd.read_csv(csv_path)
    if max_sessions is not None:
        df = df.sample(n=min(max_sessions, len(df)),
                       random_state=seed).reset_index(drop=True)

    sessions = []
    for _, row in df.iterrows():
        events  = parse_list_cell(row['Features'])
        times   = parse_list_cell(row['TimeInterval'])
        latency = (float(row['Latency'])
                   if 'Latency' in df.columns and pd.notna(row.get('Latency', float('nan')))
                   else float(sum(times) if times else 0))

        if len(events) < 2:
            continue

        # Align time intervals (Event_traces.csv has len(times) == len(events)-1)
        if len(times) == len(events) - 1:
            times = [0.0] + [float(t) for t in times]
        elif len(times) != len(events):
            times = [float(t) for t in times]
            if len(times) < len(events):
                times = times + [0.0] * (len(events) - len(times))
            else:
                times = times[:len(events)]

        sessions.append(SessionRecord(
            block_id = str(row['BlockId']),
            label    = str(row['Label']),
            events   = [str(e).strip() for e in events],
            times    = [float(t) for t in times],
            latency  = latency,
        ))
    return sessions


def split_normal_sessions(sessions: List[SessionRecord], cfg: Config):
    normal   = [s for s in sessions if safe_label_to_binary(s.label) == 0]
    abnormal = [s for s in sessions if safe_label_to_binary(s.label) == 1]
    random.shuffle(normal)
    n       = len(normal)
    n_train = int(n * cfg.train_ratio)
    n_valid = int(n * cfg.valid_ratio)
    train     = normal[:n_train]
    valid     = normal[n_train:n_train + n_valid]
    test_norm = normal[n_train + n_valid:]
    test      = test_norm + abnormal
    return train, valid, test


print('Data loading functions defined.')

Data loading functions defined.


In [10]:
# ── Dataset & Model ────────────────────────────────────────────────────────
class PathWindowDataset(Dataset):
    """Sliding-window (history -> next event) dataset for PathLSTM."""

    def __init__(self, sessions: List[SessionRecord], vocab: Vocab, history_size: int):
        self.samples = []
        pad_id = vocab.encode(Vocab.PAD)
        for session in sessions:
            encoded = [vocab.encode(e) for e in session.events]
            for i in range(1, len(encoded)):
                left = max(0, i - history_size)
                hist = encoded[left:i]
                if len(hist) < history_size:
                    hist = [pad_id] * (history_size - len(hist)) + hist
                self.samples.append((hist, encoded[i]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        hist, target = self.samples[idx]
        return (torch.tensor(hist, dtype=torch.long),
                torch.tensor(target, dtype=torch.long))


class PathLSTM(nn.Module):
    """Two-layer LSTM that predicts the next log event ID."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size  = embed_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb  = self.embedding(x)
        out, _ = self.lstm(emb)
        return self.fc(out[:, -1, :])

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print('Dataset and model classes defined.')

Dataset and model classes defined.


In [11]:
# ── Training (with MLflow per-epoch logging) ───────────────────────────────
def train_path_model(model: PathLSTM, train_loader: DataLoader,
                     valid_loader: DataLoader, cfg: Config,
                     log_mlflow: bool = True) -> Tuple:
    """Train PathLSTM; log train/valid loss per epoch to the active MLflow run."""
    device    = cfg.device
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.path_lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.7)
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_valid = float('inf')
    history    = {'train_loss': [], 'valid_loss': []}

    for epoch in range(cfg.path_epochs):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for hist, target in train_loader:
            hist, target = hist.to(device), target.to(device)
            optimizer.zero_grad()
            loss = criterion(model(hist), target)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()

        # ── Validate ───────────────────────────────────────────────────────
        model.eval()
        valid_losses = []
        with torch.no_grad():
            for hist, target in valid_loader:
                hist, target = hist.to(device), target.to(device)
                valid_losses.append(criterion(model(hist), target).item())

        mean_train = float(np.mean(train_losses))
        mean_valid = float(np.mean(valid_losses)) if valid_losses else float('inf')
        history['train_loss'].append(mean_train)
        history['valid_loss'].append(mean_valid)

        if log_mlflow:
            try:
                mlflow.log_metrics({'path_train_loss': mean_train,
                                    'path_valid_loss': mean_valid}, step=epoch)
            except Exception:
                pass

        print(f'  [Epoch {epoch+1:02d}/{cfg.path_epochs}]  '
              f'train={mean_train:.4f}  valid={mean_valid:.4f}')

        if mean_valid < best_valid:
            best_valid = mean_valid
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state:
        model.load_state_dict(best_state)
    return model, history, best_valid


print('Training function defined.')

Training function defined.


In [12]:
# ── Detection (nucleus sampling) ───────────────────────────────────────────
def detect_session(session: SessionRecord, path_model: PathLSTM,
                   vocab: Vocab, cfg: Config) -> Dict:
    """Classify one session: anomaly if any step falls outside the nucleus."""
    device = cfg.device
    pad_id = vocab.encode(Vocab.PAD)
    path_model.eval()

    ev           = [vocab.encode(e) for e in session.events]
    step_results = []

    with torch.no_grad():
        for i in range(1, len(ev)):
            left   = max(0, i - cfg.history_size)
            hist_e = ev[left:i]
            if len(hist_e) < cfg.history_size:
                hist_e = [pad_id] * (cfg.history_size - len(hist_e)) + hist_e

            hist_t     = torch.tensor(hist_e, dtype=torch.long, device=device).unsqueeze(0)
            logits     = path_model(hist_t)
            probs      = torch.softmax(logits, dim=-1).squeeze(0)
            sorted_p, sorted_ids = torch.sort(probs, descending=True)
            cumulative = torch.cumsum(sorted_p, dim=0)
            nucleus    = sorted_ids[cumulative - sorted_p < cfg.top_p].tolist()
            anomaly    = ev[i] not in nucleus

            step_results.append({
                'step_index':   i,
                'actual_event': vocab.decode(ev[i]),
                'path_anomaly': bool(anomaly),
                'nucleus_size': len(nucleus),
            })

    session_pred = int(any(x['path_anomaly'] for x in step_results))
    return {
        'BlockId':             session.block_id,
        'true_label':          safe_label_to_binary(session.label),
        'pred_label':          session_pred,
        'latency':             session.latency,
        'num_steps':           len(step_results),
        'num_anomalous_steps': sum(int(x['path_anomaly']) for x in step_results),
        'step_results':        step_results,
    }


print('Detection function defined.')

Detection function defined.


In [13]:
# ── Plot helpers ───────────────────────────────────────────────────────────
def plot_training_curve(history: Dict, cfg: Config, save_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history['train_loss'], 'o-', label='Train Loss')
    ax.plot(history['valid_loss'], 's-', label='Valid Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('CrossEntropy Loss')
    ax.set_title(f'Training Curve  (seed={cfg.seed}, top_p={cfg.top_p})')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=100)
    plt.close(fig)


def plot_confusion_matrix(y_true, y_pred, f1, save_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Anomaly'],
                yticklabels=['Normal', 'Anomaly'], ax=ax,
                annot_kws={'size': 14})
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True',      fontsize=12)
    ax.set_title(f'Confusion Matrix  (F1={f1:.4f})')
    plt.tight_layout()
    fig.savefig(save_path, dpi=100)
    plt.close(fig)


print('Plot helpers defined.')

Plot helpers defined.


In [14]:
# ── Main experiment pipeline ───────────────────────────────────────────────
def run_experiment(cfg: Config, run_name: str = None) -> Dict:
    """
    Full DeepLog pipeline with MLflow tracking.
    Returns a dict with metrics, model, vocab, and session_outputs.
    """
    set_seed(cfg.seed)

    with mlflow.start_run(run_name=run_name) as run:
        run_id     = run.info.run_id
        run_id_short = run_id[:8]
        out_dir    = ARTIFACTS_DIR / f'run_{run_id_short}'
        out_dir.mkdir(parents=True, exist_ok=True)

        # ── Log hyperparameters ────────────────────────────────────────────
        params = {k: (str(v) if v is None else v) for k, v in asdict(cfg).items()
                  if k != 'device'}
        mlflow.log_params(params)
        mlflow.set_tags({
            'model_type': 'DeepLog_LSTM',
            'dataset':    'HDFS_v1',
            'task':       'anomaly_detection',
        })

        sep = '=' * 60
        print(f'\n{sep}')
        print(f'Run: {run_id_short}  name: {run_name or ""}')
        print(f'seed={cfg.seed}, top_p={cfg.top_p}, '
              f'history={cfg.history_size}, epochs={cfg.path_epochs}, '
              f'max_sessions={cfg.max_sessions}')
        print(sep)

        # ── 1. Load data ───────────────────────────────────────────────────
        print('\n[1/5] Loading data...')
        sessions  = load_sessions(cfg.csv_path, cfg.max_sessions, cfg.seed)
        n_total   = len(sessions)
        n_normal  = sum(1 for s in sessions if safe_label_to_binary(s.label) == 0)
        n_anomaly = n_total - n_normal
        mlflow.log_metrics({'n_total': n_total, 'n_normal': n_normal, 'n_anomaly': n_anomaly})
        print(f'  Total={n_total}  Normal={n_normal}  '
              f'Anomaly={n_anomaly} ({n_anomaly/n_total*100:.1f}%)')

        # ── 2. Split ───────────────────────────────────────────────────────
        train_s, valid_s, test_s = split_normal_sessions(sessions, cfg)
        n_test_norm = sum(1 for s in test_s if safe_label_to_binary(s.label) == 0)
        n_test_anom = sum(1 for s in test_s if safe_label_to_binary(s.label) == 1)
        mlflow.log_metrics({
            'n_train':        len(train_s),
            'n_valid':        len(valid_s),
            'n_test':         len(test_s),
            'n_test_normal':  n_test_norm,
            'n_test_anomaly': n_test_anom,
        })
        print(f'  Split: train={len(train_s)}, valid={len(valid_s)}, '
              f'test={len(test_s)} (norm={n_test_norm}, anom={n_test_anom})')

        # ── 3. Vocabulary & time normalisation ────────────────────────────
        print('\n[2/5] Building vocab & normalising times...')
        train_tokens = [e for s in train_s for e in s.events]
        vocab        = Vocab(train_tokens)
        mlflow.log_metric('vocab_size', len(vocab))
        print(f'  Vocab size: {len(vocab)}')

        log1p_times = np.array([np.log1p(t) for s in train_s for t in s.times],
                               dtype=np.float32)
        time_mean = float(log1p_times.mean())
        time_std  = float(log1p_times.std() + 1e-8)

        for group in [train_s, valid_s, test_s]:
            for s in group:
                s.times = [(np.log1p(t) - time_mean) / time_std for t in s.times]

        mlflow.log_params({'time_mean': round(time_mean, 6),
                           'time_std':  round(time_std, 6)})

        # ── 4. Train PathLSTM ─────────────────────────────────────────────
        print('\n[3/5] Training PathLSTM...')
        g = torch.Generator()
        g.manual_seed(cfg.seed)

        train_ds     = PathWindowDataset(train_s, vocab, cfg.history_size)
        valid_ds     = PathWindowDataset(valid_s, vocab, cfg.history_size)
        train_loader = DataLoader(train_ds, batch_size=cfg.path_batch_size,
                                  shuffle=True, generator=g)
        valid_loader = DataLoader(valid_ds, batch_size=cfg.path_batch_size,
                                  shuffle=False)

        path_model = PathLSTM(
            vocab_size  = len(vocab),
            embed_dim   = cfg.event_embed_dim,
            hidden_dim  = cfg.path_hidden_dim,
            num_layers  = cfg.path_num_layers,
            dropout     = cfg.path_dropout,
        )
        mlflow.log_metric('model_params', path_model.count_parameters())
        print(f'  Model params: {path_model.count_parameters():,}')

        t0 = time.time()
        path_model, history, best_valid = train_path_model(
            path_model, train_loader, valid_loader, cfg, log_mlflow=True)
        train_time = time.time() - t0
        mlflow.log_metrics({'best_valid_loss': best_valid, 'train_time_s': round(train_time, 1)})
        print(f'  Training done in {train_time:.1f}s  best_valid_loss={best_valid:.4f}')

        # ── 5. Evaluate ───────────────────────────────────────────────────
        print('\n[4/5] Evaluating on test set...')
        session_outputs = []
        y_true, y_pred  = [], []
        for s in test_s:
            res = detect_session(s, path_model, vocab, cfg)
            session_outputs.append(res)
            y_true.append(res['true_label'])
            y_pred.append(res['pred_label'])

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall    = recall_score(y_true, y_pred, zero_division=0)
        f1        = f1_score(y_true, y_pred, zero_division=0)

        metrics = {
            'precision': round(float(precision), 4),
            'recall':    round(float(recall),    4),
            'f1':        round(float(f1),         4),
            'tp': int(tp), 'fp': int(fp),
            'tn': int(tn), 'fn': int(fn),
        }
        mlflow.log_metrics(metrics)
        print(f'  Precision={precision:.4f}  Recall={recall:.4f}  F1={f1:.4f}')
        print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')

        # ── 6. Artifacts ──────────────────────────────────────────────────
        print('\n[5/5] Saving artifacts...')

        # Model weights
        model_path = out_dir / 'path_model.pt'
        torch.save(path_model.state_dict(), model_path)
        mlflow.log_artifact(str(model_path), 'model')

        # Vocab
        vocab_path = out_dir / 'vocab.json'
        with open(vocab_path, 'w') as f:
            json.dump(vocab.stoi, f, indent=2)
        mlflow.log_artifact(str(vocab_path), 'model')

        # Normalisation params
        norm_path = out_dir / 'normalization.json'
        with open(norm_path, 'w') as f:
            json.dump({'time_mean': time_mean, 'time_std': time_std}, f, indent=2)
        mlflow.log_artifact(str(norm_path), 'model')

        # Config snapshot
        cfg_path = out_dir / 'config.json'
        with open(cfg_path, 'w') as f:
            json.dump(asdict(cfg), f, indent=2)
        mlflow.log_artifact(str(cfg_path), 'config')

        # Session predictions
        preds_path = out_dir / 'session_predictions.json'
        with open(preds_path, 'w') as f:
            json.dump(session_outputs, f, indent=2)
        mlflow.log_artifact(str(preds_path), 'predictions')

        # Plots
        curve_path = out_dir / 'training_curve.png'
        plot_training_curve(history, cfg, curve_path)
        mlflow.log_artifact(str(curve_path), 'plots')

        cm_path = out_dir / 'confusion_matrix.png'
        plot_confusion_matrix(y_true, y_pred, f1, cm_path)
        mlflow.log_artifact(str(cm_path), 'plots')

        # Log PyTorch model object
        mlflow.pytorch.log_model(path_model, 'pytorch_model')

        print(f'  Artifacts: {out_dir}')
        print(f'  MLflow run ID: {run_id}')

    return {
        'run_id':          run_id,
        'session_outputs': session_outputs,
        'y_true':          y_true,
        'y_pred':          y_pred,
        'vocab':           vocab,
        'path_model':      path_model,
        'history':         history,
        'out_dir':         out_dir,
        **metrics,
    }


print('run_experiment() defined. Ready to train.')

run_experiment() defined. Ready to train.


---
## 1. Data Loading & Exploratory Analysis

In [15]:
# ── EDA: Load dataset ──────────────────────────────────────────────────────
print('Loading dataset for EDA...')
all_sessions = load_sessions(str(DATA_PATH))

labels      = [safe_label_to_binary(s.label) for s in all_sessions]
n_total     = len(labels)
n_normal    = labels.count(0)
n_anomaly   = labels.count(1)
seq_lengths = [len(s.events) for s in all_sessions]
all_events  = sorted({e for s in all_sessions for e in s.events})

print(f'\n{"Dataset Statistics":─^50}')
print(f'  Total sessions : {n_total:>10,}')
print(f'  Normal         : {n_normal:>10,}  ({n_normal/n_total*100:.1f}%)')
print(f'  Anomaly        : {n_anomaly:>10,}  ({n_anomaly/n_total*100:.1f}%)')
print(f'\n  Seq length mean: {np.mean(seq_lengths):.1f}')
print(f'  Seq length med : {int(np.median(seq_lengths))}')
print(f'  Seq length max : {max(seq_lengths)}')
print(f'\n  Unique events  : {len(all_events)}')
print(f'  Events         : {all_events}')

# Load templates for reference
if TMPL_PATH.exists():
    df_tmpl = pd.read_csv(TMPL_PATH)
    print(f'\nEvent Templates ({len(df_tmpl)} total):')
    print(df_tmpl.to_string(index=False))

Loading dataset for EDA...


FileNotFoundError: [Errno 2] No such file or directory: '/content/22-team-project/HDFS_v1/preprocessed/Event_traces.csv'

In [ ]:
# EDA plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── (a) Class distribution ─────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(['Normal', 'Anomaly'], [n_normal, n_anomaly],
               color=['steelblue', 'tomato'], edgecolor='white')
for bar, v in zip(bars, [n_normal, n_anomaly]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + n_total * 0.01,
            f'{v:,}\n({v/n_total*100:.1f}%)',
            ha='center', fontsize=10)
ax.set_ylim(0, max(n_normal, n_anomaly) * 1.18)
ax.set_title('Class Distribution')
ax.set_ylabel('# Sessions')

# ── (b) Session length histogram ──────────────────────────────────────────
ax = axes[1]
lens_norm = [len(s.events) for s in all_sessions if safe_label_to_binary(s.label) == 0]
lens_anom = [len(s.events) for s in all_sessions if safe_label_to_binary(s.label) == 1]
ax.hist(lens_norm, bins=50, alpha=0.65, label='Normal',  color='steelblue', range=(0, 60))
ax.hist(lens_anom, bins=50, alpha=0.65, label='Anomaly', color='tomato',    range=(0, 60))
ax.set_title('Session Length Distribution (clipped at 60)')
ax.set_xlabel('# Events')
ax.set_ylabel('# Sessions')
ax.legend()

# ── (c) Top-15 event frequencies ─────────────────────────────────────────
ax = axes[2]
ev_counts = Counter(e for s in all_sessions for e in s.events)
top_ev    = ev_counts.most_common(15)
ev_names, ev_vals = zip(*top_ev)
ax.barh(range(len(ev_names)), ev_vals, color='steelblue')
ax.set_yticks(range(len(ev_names)))
ax.set_yticklabels(ev_names)
ax.set_title('Top 15 Event Types')
ax.set_xlabel('Total occurrences')

plt.suptitle('HDFS LogHub v1 — Exploratory Data Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
eda_path = ARTIFACTS_DIR / 'eda_overview.png'
fig.savefig(eda_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'EDA plot saved: {eda_path}')

---
## 2. Main Experiment — Full Dataset, Best Config

> **Note:** Training on the full 575K sessions on CPU takes 10–30 min.  
> For a quick demo set `max_sessions=20000` in `Config`.

In [ ]:
cfg_main = Config(
    seed         = 42,
    path_epochs  = 1,
    top_p        = 0.997,
    history_size = 20,
    # max_sessions = 20000,  # <-- uncomment for quick test
)

results_main = run_experiment(cfg_main, run_name='deeplog_PRD_seed42_full')

print('\n' + '='*60)
print('MAIN EXPERIMENT RESULTS')
print(f'  F1:        {results_main["f1"]:.4f}')
print(f'  Precision: {results_main["precision"]:.4f}')
print(f'  Recall:    {results_main["recall"]:.4f}')
print(f'  TP={results_main["tp"]}  FP={results_main["fp"]}  '
      f'TN={results_main["tn"]}  FN={results_main["fn"]}')
print('='*60)

In [ ]:
# Full classification report
print('Block-level Classification Report:')
print(classification_report(
    results_main['y_true'],
    results_main['y_pred'],
    target_names=['Normal (Success)', 'Anomaly'],
    digits=4
))

---
## 3. Error Analysis — False Positives & False Negatives

In [ ]:
outputs = results_main['session_outputs']

tp_sessions = [s for s in outputs if s['true_label'] == 1 and s['pred_label'] == 1]
tn_sessions = [s for s in outputs if s['true_label'] == 0 and s['pred_label'] == 0]
fp_sessions = [s for s in outputs if s['true_label'] == 0 and s['pred_label'] == 1]
fn_sessions = [s for s in outputs if s['true_label'] == 1 and s['pred_label'] == 0]

print('Error Analysis Summary')
print(f'  TP (anomalies detected):          {len(tp_sessions):>6}')
print(f'  TN (normal correctly classified): {len(tn_sessions):>6}')
print(f'  FP (normal flagged as anomaly):   {len(fp_sessions):>6}')
print(f'  FN (anomalies missed):            {len(fn_sessions):>6}')

# ── FP Analysis ───────────────────────────────────────────────────────────
if fp_sessions:
    fp_sorted = sorted(fp_sessions,
                       key=lambda x: x['num_anomalous_steps'] / max(x['num_steps'], 1),
                       reverse=True)
    print(f'\nTop-10 False Positives (by anomalous-step rate):')
    print(f'{"BlockId":<32} {"Steps":>6} {"AnoSteps":>9} {"Rate":>8}')
    print('-' * 60)
    for s in fp_sorted[:10]:
        rate = s['num_anomalous_steps'] / max(s['num_steps'], 1)
        print(f'{s["BlockId"][:30]:<32} {s["num_steps"]:>6} '
              f'{s["num_anomalous_steps"]:>9} {rate:>8.2%}')

# ── FN Analysis ───────────────────────────────────────────────────────────
if fn_sessions:
    print(f'\nTop-10 False Negatives (shortest session first — subtle anomalies):')
    print(f'{"BlockId":<32} {"Steps":>6} {"AnoSteps":>9} {"Rate":>8}')
    print('-' * 60)
    for s in sorted(fn_sessions, key=lambda x: x['num_steps'])[:10]:
        rate = s['num_anomalous_steps'] / max(s['num_steps'], 1)
        print(f'{s["BlockId"][:30]:<32} {s["num_steps"]:>6} '
              f'{s["num_anomalous_steps"]:>9} {rate:>8.2%}')
else:
    print('\nNo false negatives — perfect recall!')

In [ ]:
# Error visualisations (4-panel figure)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── (a) Confusion matrix ──────────────────────────────────────────────────
ax = axes[0, 0]
cm = confusion_matrix(results_main['y_true'], results_main['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'], ax=ax, annot_kws={'size': 14})
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title(f'Confusion Matrix\n'
             f'F1={results_main["f1"]:.4f}  '
             f'P={results_main["precision"]:.4f}  '
             f'R={results_main["recall"]:.4f}')

# ── (b) Anomalous-step rate per category ─────────────────────────────────
ax = axes[0, 1]
def step_rates(sessions):
    return [s['num_anomalous_steps'] / max(s['num_steps'], 1) for s in sessions]

for data, label, color in [
        (tp_sessions, f'TP (n={len(tp_sessions)})', 'steelblue'),
        (fp_sessions, f'FP (n={len(fp_sessions)})', 'orange'),
        (fn_sessions, f'FN (n={len(fn_sessions)})', 'tomato'),
    ]:
    if data:
        ax.hist(step_rates(data), bins=30, alpha=0.65, label=label, color=color)
ax.set_xlabel('Fraction of Anomalous Steps')
ax.set_ylabel('# Sessions')
ax.set_title('Step-level Anomaly Rate by Category')
ax.legend()

# ── (c) Session length boxplot ────────────────────────────────────────────
ax = axes[1, 0]
data_box   = []
labels_box = []
for grp, lbl in [(tn_sessions, 'TN'), (tp_sessions, 'TP'),
                  (fp_sessions, 'FP'), (fn_sessions, 'FN')]:
    if grp:
        data_box.append([s['num_steps'] for s in grp])
        labels_box.append(lbl)
ax.boxplot(data_box, labels=labels_box, patch_artist=True,
           boxprops=dict(facecolor='lightblue', color='navy'))
ax.set_ylabel('Session Length (# steps)')
ax.set_title('Session Length by Prediction Category')
ax.set_yscale('log')

# ── (d) Events that trigger false positives ───────────────────────────────
ax = axes[1, 1]
fp_events = [step['actual_event']
             for s in fp_sessions
             for step in s['step_results'] if step['path_anomaly']]
if fp_events:
    top_fp = Counter(fp_events).most_common(12)
    ev, cnt = zip(*top_fp)
    ax.barh(range(len(ev)), cnt, color='orange')
    ax.set_yticks(range(len(ev)))
    ax.set_yticklabels(ev)
    ax.set_xlabel('Occurrences')
    ax.set_title('Events Triggering False Positives')
else:
    ax.text(0.5, 0.5, 'No FP events', ha='center', va='center',
            transform=ax.transAxes, fontsize=14)
    ax.set_title('Events Triggering False Positives')

plt.suptitle('DeepLog — Error Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
err_path = ARTIFACTS_DIR / 'error_analysis.png'
fig.savefig(err_path, dpi=100, bbox_inches='tight')

# Append to the main MLflow run
with mlflow.start_run(run_id=results_main['run_id']):
    mlflow.log_artifact(str(err_path), 'plots')

plt.show()
print(f'Error analysis plot saved and logged: {err_path}')

In [ ]:
# Deep-dive: why are anomalies missed (FN)?
print('False Negative Deep-Dive')
print('Hypothesis: FN sessions have event sequences that also appear in normal data.')
print('='*60)

if fn_sessions:
    for s in fn_sessions[:3]:
        events_decoded = [step['actual_event'] for step in s['step_results']]
        anomalous_at   = [step['step_index']   for step in s['step_results']
                          if step['path_anomaly']]
        print(f'\nBlockId:         {s["BlockId"]}')
        print(f'Steps:           {s["num_steps"]}  '
              f'Anomalous steps: {s["num_anomalous_steps"]}')
        disp = events_decoded[:25]
        tail = '...' if len(events_decoded) > 25 else ''
        print(f'Event sequence:  {disp}{tail}')
        print(f'Anomalous at:    {anomalous_at}')
else:
    print('No false negatives — perfect recall on this run.')

---
## 4. Reproducibility Study — Multi-Seed

Три запуска с разными seed-значениями. Используется `max_sessions=20000` для ускорения; для PRD-метрик запускайте без ограничения.

In [ ]:
REPRO_SEEDS   = [42, 123, 456]
REPRO_SESSIONS = 20000   # set to None for full dataset

repro_results = []
for seed in REPRO_SEEDS:
    r = run_experiment(
        Config(seed=seed, path_epochs=1, top_p=0.997,
               max_sessions=REPRO_SESSIONS),
        run_name=f'repro_seed{seed}'
    )
    repro_results.append({
        'seed':      seed,
        'f1':        r['f1'],
        'precision': r['precision'],
        'recall':    r['recall'],
        'tp': r['tp'], 'fp': r['fp'],
        'tn': r['tn'], 'fn': r['fn'],
        'run_id':    r['run_id'],
    })
    print(f"  seed={seed}: F1={r['f1']:.4f}  P={r['precision']:.4f}  R={r['recall']:.4f}\n")

df_repro = pd.DataFrame(repro_results)
print('\nReproducibility Summary:')
display(df_repro[['seed', 'f1', 'precision', 'recall', 'fp', 'fn']])
print(f"F1:        {df_repro['f1'].mean():.4f} ± {df_repro['f1'].std():.4f}")
print(f"Precision: {df_repro['precision'].mean():.4f} ± {df_repro['precision'].std():.4f}")
print(f"Recall:    {df_repro['recall'].mean():.4f} ± {df_repro['recall'].std():.4f}")

In [ ]:
# Reproducibility bar chart
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics_list = ['f1', 'precision', 'recall']
titles       = ['F1-Score', 'Precision', 'Recall']
colors       = ['steelblue', 'seagreen', 'tomato']

for ax, metric, title, color in zip(axes, metrics_list, titles, colors):
    vals  = df_repro[metric].values
    seeds = [str(s) for s in df_repro['seed'].values]
    bars  = ax.bar(seeds, vals, color=color, alpha=0.75, edgecolor='white')
    ax.axhline(vals.mean(), color='black', linestyle='--', alpha=0.5,
               label=f'mean={vals.mean():.4f}')
    ax.set_xlabel('Random Seed')
    ax.set_ylabel(title)
    ax.set_title(f'{title}\n(σ={vals.std():.4f})')
    ax.set_ylim(max(0, vals.min() - 0.05), min(1.0, vals.max() + 0.05))
    ax.legend(fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003, f'{v:.4f}',
                ha='center', fontsize=10)

plt.suptitle('DeepLog Reproducibility Study (3 seeds)', fontsize=13, fontweight='bold')
plt.tight_layout()
repro_path = ARTIFACTS_DIR / 'reproducibility_study.png'
fig.savefig(repro_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Plot saved: {repro_path}')

---
## 5. Robustness — Threshold Sensitivity (top-p)

Анализ чувствительности метрик к параметру `top_p` (порог nucleus sampling).  
Оптимальное значение — то, при котором F1 максимален.

In [ ]:
TOP_P_VALUES  = [0.980, 0.990, 0.995, 0.997, 0.999]
THRESH_SESSIONS = 20000

thresh_results = []
for top_p in TOP_P_VALUES:
    r = run_experiment(
        Config(seed=42, path_epochs=1, top_p=top_p,
               max_sessions=THRESH_SESSIONS),
        run_name=f'threshold_topp{top_p:.3f}'
    )
    thresh_results.append({
        'top_p':     top_p,
        'f1':        r['f1'],
        'precision': r['precision'],
        'recall':    r['recall'],
        'fp':        r['fp'],
        'fn':        r['fn'],
        'run_id':    r['run_id'],
    })
    print(f"  top_p={top_p}: F1={r['f1']:.4f}  P={r['precision']:.4f}  "
          f"R={r['recall']:.4f}  FP={r['fp']}  FN={r['fn']}\n")

df_thresh = pd.DataFrame(thresh_results)
print('\nThreshold Sensitivity Summary:')
display(df_thresh[['top_p', 'f1', 'precision', 'recall', 'fp', 'fn']])

best_tp_idx = df_thresh['f1'].idxmax()
best_top_p  = df_thresh.loc[best_tp_idx, 'top_p']
print(f'\nBest top-p = {best_top_p}  (F1 = {df_thresh.loc[best_tp_idx, "f1"]:.4f})')

In [ ]:
# Threshold sensitivity plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(df_thresh['top_p'], df_thresh['f1'],        'o-', label='F1',        color='steelblue', lw=2)
ax.plot(df_thresh['top_p'], df_thresh['precision'], 's--', label='Precision', color='seagreen')
ax.plot(df_thresh['top_p'], df_thresh['recall'],    '^--', label='Recall',    color='tomato')
ax.axvline(best_top_p, color='purple', linestyle=':', alpha=0.7,
           label=f'Best top-p={best_top_p}')
ax.set_xlabel('top-p threshold')
ax.set_ylabel('Score')
ax.set_title('Metric Sensitivity to top-p\n(nucleus sampling threshold)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)

ax = axes[1]
ax.plot(df_thresh['top_p'], df_thresh['fp'], 'o-', label='False Positives', color='orange', lw=2)
ax.plot(df_thresh['top_p'], df_thresh['fn'], 's-', label='False Negatives', color='tomato',  lw=2)
ax.axvline(best_top_p, color='purple', linestyle=':', alpha=0.7,
           label=f'Best top-p={best_top_p}')
ax.set_xlabel('top-p threshold')
ax.set_ylabel('Count')
ax.set_title('FP / FN vs top-p Threshold')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle(f'DeepLog Robustness — Threshold Sensitivity  '
             f'(best top-p = {best_top_p})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
thresh_path = ARTIFACTS_DIR / 'threshold_sensitivity.png'
fig.savefig(thresh_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Threshold sensitivity plot saved: {thresh_path}')

---
## 6. MLflow Experiment Summary

In [ ]:
client = MlflowClient(tracking_uri=MLFLOW_URI)
runs   = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.f1 DESC'],
)

rows = []
for run in runs:
    m = run.data.metrics
    p = run.data.params
    rows.append({
        'Run Name':  run.info.run_name or run.info.run_id[:8],
        'F1':        round(m.get('f1',        float('nan')), 4),
        'Precision': round(m.get('precision', float('nan')), 4),
        'Recall':    round(m.get('recall',    float('nan')), 4),
        'top_p':     p.get('top_p', '-'),
        'seed':      p.get('seed',  '-'),
        'max_sess':  p.get('max_sessions', 'all'),
        'TP':        int(m.get('tp', 0)),
        'FP':        int(m.get('fp', 0)),
        'FN':        int(m.get('fn', 0)),
    })

df_all_runs = pd.DataFrame(rows)
print(f'All MLflow Runs in "{EXPERIMENT_NAME}" (sorted by F1 DESC):')
try:
    display(df_all_runs)
except Exception:
    print(df_all_runs.to_string(index=False))

# Best run
best_run = runs[0]
print(f'\nBest Run:  {best_run.info.run_name}')
print(f'  F1:      {best_run.data.metrics.get("f1", "N/A")}')
print(f'  Run ID:  {best_run.info.run_id}')
print(f'\nLaunch MLflow UI:')
print(f'  mlflow ui --backend-store-uri {MLFLOW_URI}')

In [ ]:
# All-runs comparison chart
df_viz = df_all_runs.sort_values('F1', ascending=True).tail(15)  # top-15

fig, ax = plt.subplots(figsize=(10, max(4, len(df_viz) * 0.4)))
colors  = ['steelblue' if 'PRD' in name else 'lightsteelblue'
           for name in df_viz['Run Name']]
bars = ax.barh(df_viz['Run Name'], df_viz['F1'], color=colors, edgecolor='white')
for bar, v in zip(bars, df_viz['F1']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9)
ax.set_xlabel('F1-Score')
ax.set_title('All MLflow Runs — F1 Score Comparison\n(dark = PRD/main experiment)')
ax.set_xlim(0, 1.05)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
cmp_path = ARTIFACTS_DIR / 'runs_comparison.png'
fig.savefig(cmp_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Comparison chart saved: {cmp_path}')

---
## Summary

### Результаты

| Аспект | Подробности |
|---|---|
| **Модель** | DeepLog PathLSTM (2 слоя, hidden=64, embed=16) |
| **Детекция** | Nucleus sampling (top-p), сессия аномальна если хотя бы один шаг вне ядра |
| **Данные** | HDFS LogHub v1: 575K сессий, 3.1% аномалий |
| **Обучение** | Только на нормальных сессиях (unsupervised), 1 эпоха, batch=1024 |
| **Воспроизводимость** | Фиксированный seed на всех уровнях (Python, NumPy, PyTorch, DataLoader) |

### Артефакты MLflow (на каждый run)
- `model/path_model.pt` — веса модели
- `model/vocab.json` — словарь событий
- `model/normalization.json` — параметры нормализации времени
- `config/config.json` — полная конфигурация
- `predictions/session_predictions.json` — предсказания по сессиям
- `plots/training_curve.png`, `confusion_matrix.png`, `error_analysis.png`
- `pytorch_model/` — MLflow PyTorch model format (для деплоя)

### Как запустить UI
```bash
mlflow ui --backend-store-uri MLflow/mlruns
# открыть http://127.0.0.1:5000
```